In [ ]:
!pip uninstall -y numpy
!pip install --no-cache-dir numpy==1.26.4

import numpy as np
print(np.__version__)

Found existing installation: numpy 2.4.6
Uninstalling numpy-2.4.6:
  Successfully uninstalled numpy-2.4.6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 135.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; pytho

2.0.2


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║          AgenticDocVision — Installation Cell                   ║
# ║          University of Engineering & Technology, Lahore         ║
# ║  Run this ONCE per Colab session before anything else           ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── System dependencies (Tesseract OCR engine + PDF converter) ────────
!apt-get install -y tesseract-ocr tesseract-ocr-eng poppler-utils -q
!pip install -q rank-bm25 layoutparser opencv-python scikit-learn
# ── Python libraries ──────────────────────────────────────────────────
!pip install -q \
    "numpy==1.26.4" \
    "chromadb==0.4.24" \
    pytesseract \
    opencv-python-headless \
    pdf2image \
    Pillow \
    sentence-transformers \
    langchain \
    langchain-groq \
    langchain-community \
    groq

# ── Force correct numpy version ───────────────────────────────────────
!pip install --no-cache-dir numpy==1.26.4 -q
!pip install -q ultralytics
# ── Document-layout YOLO (title/text/table/figure/formula detection) ──
!pip install -q doclayout-yolo huggingface_hub

print("✅ Installation complete!")


Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng set to manually installed.
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 3 not upgraded.
Need to get 186 kB of archives.
After this operation, 697 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.12 [186 kB]
Fetched 186 kB in 0s (1,091 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
     ━━━━━━━━━━

In [ ]:
# ════════════════════════════════════════════════════════
# SECTION 1 — IMPORTS
# All required libraries for the entire pipeline
# ════════════════════════════════════════════════════════
import os
import re
from matplotlib import pyplot as plt
from ultralytics import YOLO
import torch
import json
import copy
import shutil
import json as json_module
import numpy as np
import cv2
import pytesseract
import chromadb
import pandas as pd
from PIL import Image
from pathlib import Path
from pdf2image import convert_from_path
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from rank_bm25 import BM25Okapi

bm25_corpus = []
bm25_index = None

print("✅ Imports done")

# ════════════════════════════════════════════════════════
# SECTION 2 — CONFIGURATION
# ════════════════════════════════════════════════════════
from google.colab import userdata

GROQ_API_KEY  = userdata.get('GROQ_API_KEY')
UPLOAD_DIR    = "/content/uploads"
OUTPUT_DIR    = "/content/outputs"
VECTOR_DB_DIR = "/content/vector_db"

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL       = "llama-3.3-70b-versatile"

print("✅ Config defined")

# ════════════════════════════════════════════════════════
# SECTION 3 — MODEL LOADING
# ────────────────────────────────────────────────────────
# Agent 2 uses DocLayout-YOLO — a YOLO model trained specifically on
# document structure (title, plain text, table, figure, formula, ...).
# A generic COCO YOLO (person/car/...) is useless for documents, so we
# load a real layout model and fall back to OpenCV layout detection if
# the weights cannot be downloaded (keeps the pipeline crash-free).
# ════════════════════════════════════════════════════════
DEVICE = 0 if torch.cuda.is_available() else "cpu"

LAYOUT_BACKEND = "cv"   # "doclayout" once the model loads successfully
layout_model   = None
try:
    from doclayout_yolo import YOLOv10
    from huggingface_hub import hf_hub_download

    _layout_weights = hf_hub_download(
        repo_id="juliozhao/DocLayout-YOLO-DocStructBench",
        filename="doclayout_yolo_docstructbench_imgsz1024.pt"
    )
    layout_model   = YOLOv10(_layout_weights)
    LAYOUT_BACKEND = "doclayout"
    print("✅ DocLayout-YOLO loaded (document-structure detection)")
except Exception as _e:
    print(f"⚠️ DocLayout-YOLO unavailable ({_e}).")
    print("   → Falling back to OpenCV contour-based layout detection.")

embedder = SentenceTransformer(EMBEDDING_MODEL)
llm      = ChatGroq(api_key=GROQ_API_KEY, model=LLM_MODEL)
print(f"✅ Models loaded — Layout: {LAYOUT_BACKEND} | LLM: {LLM_MODEL} | Embeddings: {EMBEDDING_MODEL}")

# ════════════════════════════════════════════════════════
# SECTION 4 — CHROMADB INITIALIZATION
# Using EphemeralClient (in-memory) to avoid readonly errors
# ════════════════════════════════════════════════════════
chroma_client   = chromadb.EphemeralClient()
word_collection = chroma_client.get_or_create_collection("document_words")

embedding_collection = chroma_client.get_or_create_collection("document_embeddings")
print(f"✅ ChromaDB ready (in-memory) — Count: {word_collection.count()}")

# ════════════════════════════════════════════════════════
# SECTION 5 — FOLDER SETUP
# ════════════════════════════════════════════════════════
os.makedirs(UPLOAD_DIR,    exist_ok=True)
os.makedirs(OUTPUT_DIR,    exist_ok=True)
os.makedirs(VECTOR_DB_DIR, exist_ok=True)

REGION_CROP_DIR = "/content/region_crops"
os.makedirs(REGION_CROP_DIR, exist_ok=True)
visual_index = []

print(f"✅ Folders ready")


# ════════════════════════════════════════════════════════
# SECTION 6 — AGENT 1: PREPROCESSING
# ────────────────────────────────────────────────────────
# Responsibilities:
#   - Accept PDF or image file
#   - Convert PDF pages to images (300 DPI)
#   - Apply Otsu binarization (grayscale → black & white)
#   - Remove noise using morphological opening
#   - Detect connected components (individual character blobs)
#   - Save cleaned images to OUTPUT_DIR
#   - Return base doc_json with page metadata
# ════════════════════════════════════════════════════════

def agent1_preprocess(file_path: str) -> dict:
    """
    AGENT 1 — Preprocessing Agent
    ReAct pattern:
      Thought → What file type is this? How many pages?
      Action  → Convert, binarize, denoise, detect components
      Observe → Cleaned image saved, components counted
    """
    file_path       = Path(file_path)
    file_ext        = file_path.suffix.lower()
    doc_id          = file_path.stem
    page_images_raw = []

    # ── THOUGHT: Detect file type and load pages ──────────────────
    if file_ext == ".pdf":
        # Convert each PDF page to high-resolution PIL image (300 DPI)
        pil_pages = convert_from_path(str(file_path), dpi=300)
        for pil_img in pil_pages:
            # Convert PIL (RGB) → NumPy BGR (OpenCV format)
            page_images_raw.append(cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR))

    elif file_ext in [".png", ".jpg", ".jpeg", ".tiff", ".bmp"]:
        img = cv2.imread(str(file_path))
        if img is None:
            raise ValueError(f"Could not read image: {file_path}")
        page_images_raw.append(img)
    else:
        raise ValueError(f"Unsupported file type: {file_ext}. Use PDF or image.")

    pages = []
    for page_num, img in enumerate(page_images_raw, start=1):

        # ── ACTION: Convert to grayscale ──────────────────────────
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        # ── ACTION: Otsu binarization → pure black & white ────────
        # Otsu automatically finds the best threshold value
        _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # ── ACTION: Noise removal via morphological opening ────────
        # Opening = erosion then dilation
        # Removes small noise dots while preserving text strokes
        kernel = np.ones((2, 2), np.uint8)
        clean  = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

        # ── ACTION: Connected components analysis ──────────────────
        # Groups connected pixels into labeled blobs (characters)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
            clean, connectivity=8
        )

        # Filter out tiny components smaller than 50px (noise)
        filtered = np.zeros_like(clean)
        for i in range(1, num_labels):
            if stats[i, cv2.CC_STAT_AREA] >= 50:
                filtered[labels == i] = 255

        # ── OBSERVE: Save cleaned image (used by OCR — Agent 3) ────
        out_path = os.path.join(OUTPUT_DIR, f"{doc_id}_page_{page_num:03d}_clean.png")
        cv2.imwrite(out_path, filtered)

        # ── Save original page too (used for layout — Agent 2) ─────
        # Layout models work far better on the natural page than on a
        # pure black/white binary. Same dimensions → bboxes stay aligned.
        orig_path = os.path.join(OUTPUT_DIR, f"{doc_id}_page_{page_num:03d}_orig.png")
        cv2.imwrite(orig_path, img)

        h, w = filtered.shape
        pages.append({
            "page_num":       page_num,
            "page_no":        page_num,
            "image_path":     out_path,
            "original_path":  orig_path,
            "width":          w,
            "height":         h,
            "num_components": int(num_labels - 1),
            "layout_regions": []
        })
        print(f"  ✅ Page {page_num}: {num_labels-1} components → {out_path}")

    doc_json = {
        "doc_id":      doc_id,
        "document":    doc_id,
        "total_pages": len(pages),
        "pages":       pages
    }
    print(f"\n📄 Agent 1 done — {len(pages)} page(s) preprocessed for '{doc_id}'")
    return doc_json


# ════════════════════════════════════════════════════════
# SECTION 7 — AGENT 2: LAYOUT ANALYSIS
# ────────────────────────────────────────────────────────
# Responsibilities:
#   - Load each cleaned page image from Agent 1
#   - Detect layout regions using dilation + contour detection
#   - Find line boundaries using vertical histogram projection
#   - Assign global line numbers across entire page
#   - Return updated doc_json with regions and lines
# ════════════════════════════════════════════════════════
# ════════════════════════════════════════════════════════
# HELPERS (Agent 2 Core Utilities)
# ════════════════════════════════════════════════════════

def filter_noise_boxes(boxes, min_area=800):
    """Remove tiny OCR/noise regions"""
    filtered = []
    for (x, y, w, h) in boxes:
        if w * h >= min_area:
            filtered.append((x, y, w, h))
    return filtered


def iou_merge(box1, box2, threshold=0.3):
    """Check if two boxes should be merged (overlap logic)"""
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    xi1 = max(x1, x2)
    yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2)
    yi2 = min(y1 + h1, y2 + h2)

    if xi2 <= xi1 or yi2 <= yi1:
        return False

    inter_area = (xi2 - xi1) * (yi2 - yi1)
    box1_area = w1 * h1
    box2_area = w2 * h2

    iou = inter_area / float(box1_area + box2_area - inter_area)
    return iou > threshold


def merge_boxes(boxes):
    """Merge overlapping/nearby bounding boxes"""
    merged = []

    for box in boxes:
        x, y, w, h = box
        added = False

        for i in range(len(merged)):
            if iou_merge(merged[i], box):
                mx, my, mw, mh = merged[i]

                nx = min(mx, x)
                ny = min(my, y)
                nw = max(mx + mw, x + w) - nx
                nh = max(my + mh, y + h) - ny

                merged[i] = (nx, ny, nw, nh)
                added = True
                break

        if not added:
            merged.append(box)

    return merged


def sort_reading_order(regions):
    """Top-to-bottom, left-to-right sorting"""
    return sorted(regions, key=lambda r: (r["bbox"][1], r["bbox"][0]))


def estimate_text_density(binary_crop):
    """Estimate how much text is inside a region"""
    return np.count_nonzero(binary_crop == 255) / binary_crop.size


def _box_iou(a, b):
    """IoU + containment ratio for two [x, y, w, h] boxes."""
    ax, ay, aw, ah = a
    bx, by, bw, bh = b
    x1, y1 = max(ax, bx), max(ay, by)
    x2, y2 = min(ax + aw, bx + bw), min(ay + ah, by + bh)
    if x2 <= x1 or y2 <= y1:
        return 0.0, 0.0
    inter = (x2 - x1) * (y2 - y1)
    a_area, b_area = aw * ah, bw * bh
    iou      = inter / float(a_area + b_area - inter)
    contain  = inter / float(min(a_area, b_area) + 1e-6)  # smaller box coverage
    return iou, contain


def suppress_overlapping_regions(regions, iou_thresh=0.55, contain_thresh=0.80):
    """
    Remove duplicate/overlapping layout regions so the SAME text is never
    OCR'd twice. Keeps the larger region; drops any later region that
    heavily overlaps (IoU) or is mostly contained inside a kept one.
    """
    ordered = sorted(regions, key=lambda r: r["bbox"][2] * r["bbox"][3],
                     reverse=True)
    kept = []
    for r in ordered:
        duplicate = False
        for k in kept:
            iou, contain = _box_iou(r["bbox"], k["bbox"])
            if iou > iou_thresh or contain > contain_thresh:
                duplicate = True
                break
        if not duplicate:
            kept.append(r)
    return kept


# ════════════════════════════════════════════════════════
# AGENT 2 — DOCUMENT LAYOUT ANALYSIS
# ────────────────────────────────────────────────────────
# Primary  : DocLayout-YOLO → semantic regions
#            (title, plain text, table, figure, formula, caption, ...)
# Fallback : OpenCV dilation + contour grouping → text blocks
# Both paths return regions sorted in natural reading order.
# ════════════════════════════════════════════════════════

# DocLayout "abandon" = page furniture (running heads, footers, page
# numbers) — not real content, so we drop it from the index.
_LAYOUT_DROP = {"abandon"}


def _doclayout_regions(img):
    """Detect regions with DocLayout-YOLO. Returns a list of region dicts."""
    det = layout_model.predict(
        img, imgsz=1024, conf=0.2, device=DEVICE, verbose=False
    )[0]

    names = layout_model.names
    regions = []

    for box in det.boxes:
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])
        label  = names[cls_id] if isinstance(names, (list, tuple)) else names.get(cls_id, str(cls_id))

        if label in _LAYOUT_DROP:
            continue

        x1, y1, x2, y2 = map(int, box.xyxy[0])
        x1, y1 = max(0, x1), max(0, y1)
        if x2 <= x1 or y2 <= y1:
            continue

        regions.append({
            "bbox": [x1, y1, x2 - x1, y2 - y1],
            "region_type": label,
            "confidence": round(conf, 3),
            "words_found": 0
        })

    return regions


def _cv_layout_regions(img):
    """OpenCV fallback: group text into blocks via dilation + contours."""
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    H, W = gray.shape[:2]

    # Text → white on black so we can dilate words into blocks.
    _, th = cv2.threshold(gray, 0, 255,
                          cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (25, 12))
    dilated = cv2.dilate(th, kernel, iterations=2)

    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL,
                                   cv2.CHAIN_APPROX_SIMPLE)
    boxes = [cv2.boundingRect(c) for c in contours]
    boxes = filter_noise_boxes(boxes, min_area=800)
    boxes = merge_boxes(boxes)

    regions = []
    for (x, y, w, h) in boxes:
        crop    = th[y:y + h, x:x + w]
        density = estimate_text_density(crop)
        ar      = w / float(h + 1e-5)

        if h > 0.40 * H and density < 0.15:
            rtype = "figure"
        elif ar > 6 and h < 0.05 * H:
            rtype = "title"
        else:
            rtype = "plain text"

        regions.append({
            "bbox": [x, y, w, h],
            "region_type": rtype,
            "confidence": round(float(density), 3),
            "words_found": 0
        })

    return regions


def agent2_layout_analysis(doc_json: dict) -> dict:
    result_json = copy.deepcopy(doc_json)

    print(f"\n🔵 Agent 2 — Layout Analysis (backend: {LAYOUT_BACKEND})")
    print("=" * 60)

    for page in result_json["pages"]:
        # Layout runs on the original page; OCR later uses the clean binary.
        layout_src = page.get("original_path") or page["image_path"]
        img = cv2.imread(layout_src)
        if img is None:
            img = cv2.imread(page["image_path"])
        if img is None:
            page["layout_regions"] = []
            print(f"   ⚠️ Page {page['page_no']}: image unreadable, skipped")
            continue

        regions = []
        if LAYOUT_BACKEND == "doclayout" and layout_model is not None:
            try:
                regions = _doclayout_regions(img)
            except Exception as e:
                print(f"   ⚠️ DocLayout failed on page {page['page_no']} ({e}); using CV")
                regions = []

        # Fallback (or backup if the model found nothing on this page).
        if not regions:
            regions = _cv_layout_regions(img)

        # Drop overlapping/duplicate regions so text isn't OCR'd twice.
        regions = suppress_overlapping_regions(regions)

        # Sort top→bottom, left→right and assign stable reading-order ids.
        regions = sort_reading_order(regions)
        for i, r in enumerate(regions, start=1):
            r["region_id"] = i

        page["layout_regions"] = regions
        print(f"   ✅ Page {page['page_no']}: {len(regions)} regions detected")

    return result_json


# ════════════════════════════════════════════════════════
# SECTION 8 — AGENT 3: OCR + INDEXING
# ────────────────────────────────────────────────────────
# Responsibilities:
#   - Loop through every page → region → line from Agent 2
#   - Crop each line from the full page image
#   - Run Tesseract OCR on each line crop
#   - Extract word text, position (bbox), confidence score
#   - Extract character-level bounding boxes
#   - Store each word in ChromaDB with full metadata
#   - Return doc_json with words filled in
# ════════════════════════════════════════════════════════
global bm25_corpus

def uid_safe(name: str) -> str:
    """Sanitize a document name into a filesystem-safe slug."""
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(name)).strip("_") or "doc"


def _compute_orb_descriptor(gray_crop: np.ndarray):
    """
    Compute fixed-length ORB descriptor for a region crop.
    Used by Agent 3 (indexing) and Agent 5 (image query).
    Returns float32 numpy array of 512 values, or None.
    """
    resized  = cv2.resize(gray_crop, (128, 64), interpolation=cv2.INTER_AREA)
    orb      = cv2.ORB_create(nfeatures=256)
    _, descs = orb.detectAndCompute(resized, None)
    if descs is None or len(descs) == 0:
        return None
    flat  = descs.flatten().astype(np.float32)
    fixed = np.zeros(512, dtype=np.float32)
    n     = min(len(flat), 512)
    fixed[:n] = flat[:n]
    return fixed

def _orb_match_score(desc1: np.ndarray, desc2: np.ndarray) -> float:
    """
    Compare two ORB descriptors. Returns similarity score 0.0 to 1.0.
    Higher = more visually similar.
    """
    d1 = desc1[:512].astype(np.uint8).reshape(-1, 32)
    d2 = desc2[:512].astype(np.uint8).reshape(-1, 32)
    if d1.shape[0] == 0 or d2.shape[0] == 0:
        return 0.0
    bf      = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(d1, d2)
    if not matches:
        return 0.0
    avg_dist = sum(m.distance for m in matches) / len(matches)
    score    = max(0.0, 1.0 - avg_dist / 256.0)
    coverage = min(len(matches) / max(d1.shape[0], d2.shape[0]), 1.0)
    return round(score * 0.7 + coverage * 0.3, 4)

def _template_match_score(query_gray: np.ndarray,
                           region_crop_path: str) -> float:
    """
    Template matching fallback — best for near-exact crops.
    Returns similarity score 0.0 to 1.0.
    """
    region_img = cv2.imread(region_crop_path, cv2.IMREAD_GRAYSCALE)
    if region_img is None:
        return 0.0
    qh, qw = query_gray.shape[:2]
    rh, rw = region_img.shape[:2]
    if qw > rw or qh > rh:
        template = cv2.resize(query_gray, (rw, rh))
        source   = region_img
    else:
        template = query_gray
        source   = region_img
    result   = cv2.matchTemplate(source, template, cv2.TM_CCOEFF_NORMED)
    _, max_val, _, _ = cv2.minMaxLoc(result)
    return round(float(max_val), 4)


def agent3_ocr_and_index(doc_json: dict) -> dict:
    global word_collection, embedding_collection, visual_index

    result_json = copy.deepcopy(doc_json)
    doc_name = result_json["doc_id"]

    visual_index.clear()

    print("\n🟣 Agent 3 — OCR + Indexing (FIXED SAFE VERSION)")

    for page in result_json["pages"]:
        page_no = page["page_no"]
        img = cv2.imread(page["image_path"])
        if img is None:
            continue

        for region in page["layout_regions"]:
            x, y, w, h = region["bbox"]
            crop = img[y:y+h, x:x+w]
            if crop.size == 0:
                region["text"] = ""
                continue

            gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)

            data = pytesseract.image_to_data(
                gray,
                output_type=pytesseract.Output.DICT,
                config="--oem 3 --psm 6"
            )

            region_words = []

            for i in range(len(data["text"])):
                text = data["text"][i].strip()
                conf = float(data["conf"][i])

                if text == "" or conf < 60:
                    continue

                # Word bbox in full-page coordinates → store as
                # left/top/right/bottom (the keys Agent 5 reads).
                wx = x + data["left"][i]
                wy = y + data["top"][i]
                ww = data["width"][i]
                wh = data["height"][i]

                uid = f"{doc_name}__p{page_no}__r{region['region_id']}__w{i}"

                meta = {
                    "document":    doc_name,
                    "book":        doc_name,
                    "page_no":     page_no,
                    "region_id":   region["region_id"],
                    "region_type": region.get("region_type", ""),
                    "line_no":     int(data["line_num"][i]),
                    "word_no":     int(data["word_num"][i]),
                    "text":        text,
                    "confidence":  round(conf, 1),
                    "left":        int(wx),
                    "top":         int(wy),
                    "right":       int(wx + ww),
                    "bottom":      int(wy + wh),
                }

                # STORE WORD (source of truth)
                word_collection.upsert(
                    ids=[uid],
                    documents=[text],
                    metadatas=[meta]
                )

                # STORE EMBEDDING (semantic search)
                embedding_collection.upsert(
                    ids=[uid],
                    documents=[text],
                    embeddings=[embedder.encode(text).tolist()],
                    metadatas=[meta]
                )

                region_words.append(text)

            region["text"] = " ".join(region_words)
            region["words_found"] = len(region_words)

            # ── Build the VISUAL INDEX (powers image search in Agent 5) ──
            desc = _compute_orb_descriptor(gray)
            if desc is not None:
                crop_path = os.path.join(
                    REGION_CROP_DIR, f"{uid_safe(doc_name)}_p{page_no}_r{region['region_id']}.png"
                )
                cv2.imwrite(crop_path, crop)
                visual_index.append({
                    "book":       doc_name,
                    "page_no":    page_no,
                    "region_id":  region["region_id"],
                    "bbox":       [x, y, x + w, y + h],
                    "descriptor": desc,
                    "crop_path":  crop_path,
                })

    print(f"✅ OCR + indexing completed — {word_collection.count()} words, "
          f"{len(visual_index)} visual regions")
    return result_json


# ════════════════════════════════════════════════════════
# SECTION 9 — AGENT 4: INPUT CORRECTION (LLM / ReAct)
# ────────────────────────────────────────────────────────
# Responsibilities:
#   - Accept raw user query (may have typos/OCR errors)
#   - Send to Groq LLM (Llama 3.3) with domain context
#   - LLM corrects typos, spacing, capitalization
#   - Self-corrects if LLM response is malformed (retry)
#   - Returns corrected query + reason + confidence
# ════════════════════════════════════════════════════════

def _extract_json(raw: str) -> dict:
    """Robustly pull a JSON object out of an LLM response."""
    if not raw:
        raise ValueError("empty response")
    raw = raw.strip()
    # Strip markdown code fences if present.
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw).strip()
    # Grab the first {...} block (handles any stray prose around it).
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        raw = match.group(0)
    return json.loads(raw)


# Focused, OCR-specific correction prompt. Tight rules keep the model from
# rewriting or hallucinating content — it only repairs scanning artifacts.
CORRECTION_SYSTEM_PROMPT = """You are an OCR post-correction engine for scanned documents.
You receive raw text extracted by Tesseract from a single document region.
Your ONLY job is to repair OCR scanning errors — not to rewrite the text.

FIX these common OCR mistakes:
- Character confusions: rn↔m, cl↔d, 0↔O, 1↔l↔I, 5↔S, 8↔B, vv↔w, |↔I.
- Words wrongly split ("inform ation" → "information") or merged ("ofthe" → "of the").
- Stray punctuation/symbols inserted mid-word, and broken spacing around punctuation.
- Obvious misspellings that are clearly scan artifacts.

STRICT RULES:
- Do NOT add, remove, summarize, translate, or reorder words.
- Do NOT change correct technical terms, proper nouns, numbers, codes, or units.
- Preserve the original language and capitalization intent.
- If a token is uncertain or already valid, leave it unchanged.
- Keep the meaning and word count essentially identical.

Respond with ONLY a JSON object, no markdown and no commentary:
{"corrected_text": "<the corrected text>"}"""


def agent4_correct_ocr(doc_json: dict) -> dict:
    print("\n🟡 Agent 4 — LLM OCR Correction (JSON mode, safe — no DB write)")

    result_json = copy.deepcopy(doc_json)

    # Force Groq into guaranteed-valid JSON output.
    try:
        corrector = llm.bind(response_format={"type": "json_object"})
    except Exception:
        corrector = llm

    fixed_regions = 0

    for page in result_json["pages"]:
        for region in page["layout_regions"]:

            words = (region.get("text") or "").strip()
            if not words:
                region["corrected_text"] = ""
                continue

            corrected = words  # safe default = original OCR text

            # Up to 2 attempts to obtain valid JSON before giving up.
            for _attempt in range(2):
                try:
                    response = corrector.invoke([
                        SystemMessage(content=CORRECTION_SYSTEM_PROMPT),
                        HumanMessage(content=json.dumps({"ocr_text": words}))
                    ])
                    data = _extract_json(response.content)
                    value = data.get("corrected_text")
                    if isinstance(value, str) and value.strip():
                        corrected = value.strip()
                        fixed_regions += 1
                        break
                except Exception:
                    continue  # retry, then fall back to original

            region["corrected_text"] = corrected

    print(f"✅ OCR correction done — {fixed_regions} region(s) corrected (safe mode)")
    return result_json



# ════════════════════════════════════════════════════════
# SECTION 10 — AGENT 5: RAG RETRIEVAL (UPDATED)
# ────────────────────────────────────────────────────────
# search_type = "word"     → exact/fuzzy word match (unchanged)
# search_type = "line"     → line grouping (unchanged)
# search_type = "semantic" → embedding similarity search (NEW)
# ════════════════════════════════════════════════════════

from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()


def _pack(query, results, search_type):
    """Uniform response envelope for every search mode."""
    return {
        "query":       query,
        "results":     results,
        "type":        search_type,
        "search_type": search_type,
        "total_found": len(results),
    }


def _meta_bbox(meta):
    return [meta.get("left", 0), meta.get("top", 0),
            meta.get("right", 0), meta.get("bottom", 0)]


def dedup_rows(rows, x_tol=25, y_tol=15):
    """
    Collapse duplicate hits caused by overlapping layout regions.
    Two rows are 'the same hit' when they are the same word, on the same
    page, at essentially the same pixel location (region_id ignored).
    Proximity-based (not grid-bucketed) so small pixel jitter between
    overlapping regions can't split a duplicate across bucket edges.
    """
    kept = []
    for r in rows:
        bx = r.get("bbox") or [0, 0, 0, 0]
        w  = str(r.get("word", "")).lower().strip()
        is_dup = False
        for k in kept:
            kb = k.get("bbox") or [0, 0, 0, 0]
            if (k.get("book", "")  == r.get("book", "")
                and k.get("page_no", 0) == r.get("page_no", 0)
                and str(k.get("word", "")).lower().strip() == w
                and abs(kb[0] - bx[0]) <= x_tol
                and abs(kb[1] - bx[1]) <= y_tol):
                is_dup = True
                break
        if not is_dup:
            kept.append(r)
    return kept


def ocr_query_image(path: str) -> str:
    """
    Read the text out of an uploaded query image (a screenshot/crop of a
    word or phrase). This is what makes image search find the actual WORD
    instead of doing fuzzy visual feature matching. Returns "" if no text.
    """
    img = cv2.imread(path)
    if img is None:
        return ""

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # Upscale small crops — Tesseract is far more accurate on larger text.
    h, w = gray.shape[:2]
    if max(h, w) < 600:
        scale = 600.0 / max(h, w)
        gray  = cv2.resize(gray, (int(w * scale), int(h * scale)),
                           interpolation=cv2.INTER_CUBIC)

    _, th = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # psm 7 = single text line; fall back to psm 6 (block) if empty.
    for psm in (7, 6):
        text = pytesseract.image_to_string(
            th, config=f"--oem 3 --psm {psm}"
        ).strip()
        if text:
            return " ".join(text.split())  # normalise whitespace/newlines
    return ""


def _row_from_meta(doc, meta, score):
    """Build a frontend-ready result row from stored word metadata."""
    return {
        "word":       doc,
        "score":      round(float(score), 3),
        "book":       meta.get("book", meta.get("document", "")),
        "page_no":    meta.get("page_no", 0),
        "line_no":    meta.get("line_no", 0),
        "word_no":    meta.get("word_no", 0),
        "region_id":  meta.get("region_id", 0),
        "confidence": meta.get("confidence", 0),
        "bbox":       _meta_bbox(meta),
    }


def agent5_rag_retrieve(query, search_type="word"):
    global word_collection, embedding_collection, visual_index

    print(f"\n🔎 Search: {search_type} | Query: {query}")

    # =========================
    # WORD SEARCH (exact + close fuzzy match)
    # =========================
    if search_type == "word":
        data = word_collection.get(include=["documents", "metadatas"])
        results = []
        q = query.lower().strip()

        for doc, meta in zip(data["documents"], data["metadatas"]):
            sim = similarity(doc, q)
            if doc.lower() == q or sim >= 0.85:
                score = 1.0 if doc.lower() == q else sim
                results.append(_row_from_meta(doc, meta, score))

        results.sort(key=lambda r: -r["score"])
        return _pack(query, results, "word")

    # =========================
    # LINE SEARCH (group words by region/line, return matching lines)
    # =========================
    if search_type == "line":
        data = word_collection.get(include=["documents", "metadatas"])

        grouped = {}   # key → {"words": [...], "meta": first_meta}
        for doc, meta in zip(data["documents"], data["metadatas"]):
            key = (meta.get("book", meta.get("document", "")),
                   meta.get("page_no", 0),
                   meta.get("region_id", 0),
                   meta.get("line_no", 0))
            entry = grouped.setdefault(key, {"words": [], "meta": meta})
            entry["words"].append(doc)

        q = query.lower().strip()
        results = []
        for entry in grouped.values():
            line_text = " ".join(entry["words"])
            if q and q not in line_text.lower():
                continue
            meta = entry["meta"]
            results.append({
                "word":       line_text,          # full line in the "match" column
                "text":       line_text,
                "score":      1.0,
                "book":       meta.get("book", meta.get("document", "")),
                "page_no":    meta.get("page_no", 0),
                "line_no":    meta.get("line_no", 0),
                "word_no":    0,
                "region_id":  meta.get("region_id", 0),
                "confidence": meta.get("confidence", 0),
                "bbox":       _meta_bbox(meta),
            })

        results.sort(key=lambda r: (r["page_no"], r["region_id"], r["line_no"]))
        return _pack(query, results, "line")

    # =========================
    # SEMANTIC SEARCH (embedding similarity)
    # =========================
    if search_type == "semantic":
        if embedding_collection.count() == 0:
            return _pack(query, [], "semantic")

        q_emb = embedder.encode(query).tolist()
        res = embedding_collection.query(
            query_embeddings=[q_emb],
            n_results=min(10, embedding_collection.count()),
            include=["documents", "metadatas", "distances"]
        )

        results = []
        for doc, meta, dist in zip(
            res["documents"][0], res["metadatas"][0], res["distances"][0]
        ):
            score = max(0.0, 1 - dist / 2)   # cosine distance → similarity
            results.append(_row_from_meta(doc, meta, score))

        return _pack(query, results, "semantic")

    # =========================
    # IMAGE SEARCH (ORB descriptor match against the visual index)
    # =========================
    if search_type == "image":
        q_img = cv2.imread(query, cv2.IMREAD_GRAYSCALE)
        if q_img is None:
            return _pack(query, [], "image")
        q_img  = cv2.resize(q_img, (128, 64))
        q_desc = _compute_orb_descriptor(q_img)
        if q_desc is None:
            return _pack(query, [], "image")

        results = []
        for entry in visual_index:
            score = _orb_match_score(q_desc, entry["descriptor"])
            if score > 0.3:
                results.append({
                    "word":       "",
                    "score":      round(float(score), 3),
                    "book":       entry.get("book", ""),
                    "page_no":    entry["page_no"],
                    "line_no":    0,
                    "word_no":    0,
                    "region_id":  entry.get("region_id", 0),
                    "confidence": round(float(score) * 100, 1),
                    "bbox":       entry["bbox"],
                })

        results.sort(key=lambda x: -x["score"])
        return _pack(query, results[:20], "image")

    raise ValueError("Invalid search_type")


# ════════════════════════════════════════════════════════
# DISPLAY — updated to show semantic score label
# ════════════════════════════════════════════════════════
def show_word_crop(result):
    """
    Displays the actual word crop using stored bbox.
    """

    page_path = os.path.join(
        OUTPUT_DIR,
        f"{result['book']}_page_{result['page_no']:03d}_clean.png"
    )

    img = cv2.imread(page_path)

    if img is None:
        print("⚠️ Page image not found.")
        return

    left, top, right, bottom = result["bbox"]

    pad = 8

    x1 = max(0, left - pad)
    y1 = max(0, top - pad)
    x2 = min(img.shape[1], right + pad)
    y2 = min(img.shape[0], bottom + pad)

    crop = img[y1:y2, x1:x2]

    plt.figure(figsize=(4,2))
    plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

def display_results(results):
    print("\n" + "="*70)

    if results["search_type"] == "semantic":
        print(f"🧠 SEMANTIC SEARCH RESULTS — '{results['query']}'")
    else:
        print(f"📋 SEARCH RESULTS — '{results['query']}'")

    print("="*70)

    if results["total_found"] == 0:
        print("❌ No results found")
        return

    for i, r in enumerate(results["results"][:20], 1):
        if "text" in r:
            # line mode
            print(f"{i}. [{r['book']}] P{r['page_no']} → {r['text']}")
        else:
            # word or semantic mode
            label = "semantic score" if results["search_type"] == "semantic" else "score"
            print(f"{i}. [{r['book']}] P{r['page_no']} → '{r['word']}' ({label}: {r['score']})")
            show_word_crop(r)


# ════════════════════════════════════════════════════════
# ORCHESTRATOR — updated run_query_pipeline
# ════════════════════════════════════════════════════════
def run_query_pipeline(query, search_type="word"):
    results = agent5_rag_retrieve(query, search_type)

    print("\n================ RESULTS ================")
    for r in results["results"][:10]:
        print(r)

    return results


print()
print("✅ Agent 5 updated — 3 search types available!")
print()
print("   📌 USAGE:")
print("   ─────────────────────────────────────────────────────────")
print("   run_query_pipeline('student',   search_type='word')")
print("   run_query_pipeline('student',   search_type='line')")
print("   run_query_pipeline('student',   search_type='semantic')")
print("   ─────────────────────────────────────────────────────────")

# ════════════════════════════════════════════════════════
# SECTION 11 — ORCHESTRATOR
# ────────────────────────────────────────────────────────
# Chains all 5 agents into two easy-to-call functions:
#   run_processing_pipeline(file_path) → processes document
#   run_query_pipeline(query, type)    → searches document
# ════════════════════════════════════════════════════════

def run_processing_pipeline(file_path: str) -> dict:

    global chroma_client, word_collection, embedding_collection, visual_index

    print("🔄 Resetting system...")

    shutil.rmtree(OUTPUT_DIR, ignore_errors=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Fresh in-memory DB + visual index for each document.
    chroma_client        = chromadb.EphemeralClient()
    word_collection      = chroma_client.get_or_create_collection("document_words")
    embedding_collection = chroma_client.get_or_create_collection("document_embeddings")
    visual_index = []

    print("\n🚀 START PIPELINE")

    # 1
    doc_json = agent1_preprocess(file_path)

    # 2
    doc_json = agent2_layout_analysis(doc_json)

    # 3
    doc_json = agent3_ocr_and_index(doc_json)

    # 4 (SAFE LLM correction ONLY)
    doc_json = agent4_correct_ocr(doc_json)

    print("\n✅ COMPLETE PIPELINE FINISHED")
    print("📊 Words indexed:", word_collection.count())

    return doc_json

print()
print("✅ AgenticDocVision — All agents and orchestrator loaded!")
print()
print("   📌 USAGE:")
print("   ─────────────────────────────────────────────────────────")
print("   Step 1 → Upload PDF to /content/uploads/")
print("   Step 2 → run_processing_pipeline('/content/uploads/file.pdf')")
print("   Step 3 → run_query_pipeline('word', search_type='word')")
print("            run_query_pipeline('word', search_type='line')")

✅ Imports done
✅ Config defined
✅ DocLayout-YOLO loaded (document-structure detection)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ Models loaded — Layout: doclayout | LLM: llama-3.3-70b-versatile | Embeddings: all-MiniLM-L6-v2
✅ ChromaDB ready (in-memory) — Count: 0
✅ Folders ready

✅ Agent 5 updated — 3 search types available!

   📌 USAGE:
   ─────────────────────────────────────────────────────────
   run_query_pipeline('student',   search_type='word')
   run_query_pipeline('student',   search_type='line')
   run_query_pipeline('student',   search_type='semantic')
   ─────────────────────────────────────────────────────────

✅ AgenticDocVision — All agents and orchestrator loaded!

   📌 USAGE:
   ─────────────────────────────────────────────────────────
   Step 1 → Upload PDF to /content/uploads/
   Step 2 → run_processing_pipeline('/content/uploads/file.pdf')
   Step 3 → run_query_pipeline('word', search_type='word')
            run_query_pipeline('word', search_type='line')


In [ ]:
# ── REAL NUCLEAR RESET ──
import chromadb
import gc

# 1. Pehle se chal rahe client se collections delete karne ki koshish karein
try:
    if 'chroma_client' in globals():
        print("🗑️ Deleting old collections from memory...")
        chroma_client.delete_collection("document_words")
        chroma_client.delete_collection("document_embeddings")
except Exception:
    pass  # Agar pehle se load nahi hain toh skip karein

# 2. Force Python garbage collection to clear stuck memory references
gc.collect()

# 3. Create a completely fresh Ephemeral Client
chroma_client        = chromadb.EphemeralClient()
word_collection      = chroma_client.get_or_create_collection("document_words")
embedding_collection = chroma_client.get_or_create_collection("document_embeddings")
visual_index = []

print(f"✅ ChromaDB reset! Count: {word_collection.count()}")  # Ab guaranteed 0 aayega!

🗑️ Deleting old collections from memory...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given



🔎 Search: word | Query: student

🔎 Search: word | Query: student

🔎 Search: word | Query: student
✅ ChromaDB reset! Count: 0


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║         AgenticDocVision — RUN CELL                             ║
# ║         Change FILE_PATH and QUERIES below, then run            ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── STEP 1: Set your uploaded file path here ──────────────────────────
FILE_PATH = "/content/uploads/CSC414 Enterprise Application Development.pdf"   # ← change to your file name

# ── STEP 2: Process the document (Agents 1 → 2 → 3) ──────────────────
doc_json = run_processing_pipeline(FILE_PATH)

# ── STEP 3: Search queries (Agents 4 → 5) ────────────────────────────
# Word search — finds exact word + its location
run_query_pipeline("manipulation",  search_type="word")

# Line search — finds full lines containing the word
run_query_pipeline("knowledge",  search_type="line")

# Typo test — Agent 4 corrects before searching
run_query_pipeline("information",  search_type="semantic")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║     AgenticDocVision — Beautiful Custom Frontend (Final)        ║
# ║     Word + Line + Semantic + Image Query                        ║
# ╚══════════════════════════════════════════════════════════════════╝

!pip install fastapi uvicorn pyngrok python-multipart -q

import nest_asyncio
nest_asyncio.apply()

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.responses import HTMLResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok
import uvicorn
import shutil, os, threading

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"]
)

state = {"ready": False, "filename": "", "word_count": 0, "pages": 0}

# ══════════════════════════════════════════════════════════════════════
# API ROUTES
# ══════════════════════════════════════════════════════════════════════

@app.post("/upload")
async def upload_document(file: UploadFile = File(...)):
    try:
        file_path = f"/content/uploads/{file.filename}"
        with open(file_path, "wb") as f:
            shutil.copyfileobj(file.file, f)
        doc_json = run_processing_pipeline(file_path)
        state["ready"]      = True
        state["filename"]   = file.filename
        state["word_count"] = word_collection.count()
        state["pages"]      = doc_json["total_pages"]
        return JSONResponse({
            "success":    True,
            "filename":   file.filename,
            "pages":      doc_json["total_pages"],
            "word_count": word_collection.count()
        })
    except Exception as e:
        return JSONResponse({"success": False, "error": str(e)}, status_code=500)


@app.post("/search")
async def search(query: str = Form(...), search_type: str = Form(...)):
    if not state["ready"]:
        return JSONResponse({"success": False, "error": "No document processed yet."})
    if not query.strip():
        return JSONResponse({"success": False, "error": "Empty query."})
    try:
        # semantic uses full phrase; word/line splits into words
        if search_type == "semantic":
            result  = run_query_pipeline(query.strip(), search_type="semantic")
            unique  = dedup_rows(result["results"])
        else:
            stype   = "word" if search_type == "word" else "line"
            words   = query.strip().split()
            all_res = []
            for word in words:
                result = run_query_pipeline(word.strip(), search_type=stype)
                all_res.extend(result["results"])
            unique = dedup_rows(all_res)
            unique = sorted(unique,
                            key=lambda x: (x["book"], x["page_no"],
                                           x["bbox"][1] if x.get("bbox") else 0,
                                           x["bbox"][0] if x.get("bbox") else 0))
        return JSONResponse({
            "success":     True,
            "query":       query,
            "search_type": search_type,
            "total_found": len(unique),
            "results":     unique
        })
    except Exception as e:
        return JSONResponse({"success": False, "error": str(e)}, status_code=500)


@app.post("/image-search")
async def image_search(file: UploadFile = File(...)):
    if not state["ready"]:
        return JSONResponse({"success": False,
                             "error": "No document processed yet."})
    try:
        query_path = f"/content/uploads/query_{file.filename}"
        with open(query_path, "wb") as f:
            shutil.copyfileobj(file.file, f)

        # ── STEP 1: try to READ the word(s) in the uploaded image ──────
        extracted = ocr_query_image(query_path)

        if extracted:
            # Image contains text → find that actual WORD in the document.
            rows = []
            for tok in extracted.split():
                tok = tok.strip()
                if len(tok) < 2:
                    continue
                r = agent5_rag_retrieve(tok, search_type="word")
                rows.extend(r["results"])
            rows = dedup_rows(rows)
            rows = sorted(rows, key=lambda x: (
                x["page_no"],
                x["bbox"][1] if x.get("bbox") else 0,
                x["bbox"][0] if x.get("bbox") else 0,
            ))
            return JSONResponse({
                "success":        True,
                "search_type":    "word",       # render as real word hits
                "extracted_text": extracted,
                "total_found":    len(rows),
                "results":        rows
            })

        # ── STEP 2: no readable text (logo/figure) → visual ORB match ──
        results = agent5_rag_retrieve(query_path, search_type="image")
        return JSONResponse({
            "success":        True,
            "search_type":    "image",
            "extracted_text": "",
            "total_found":    results["total_found"],
            "results":        results["results"]
        })
    except Exception as e:
        return JSONResponse({"success": False, "error": str(e)}, status_code=500)


# ══════════════════════════════════════════════════════════════════════
# HTML FRONTEND
# ══════════════════════════════════════════════════════════════════════

HTML = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>AgenticDocVision</title>
<link href="https://fonts.googleapis.com/css2?family=DM+Serif+Display:ital@0;1&family=DM+Sans:wght@300;400;500;600&display=swap" rel="stylesheet">
<style>
  *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
  :root {
    --cream:#FAFAF7; --white:#FFFFFF; --ink:#1A1A1A; --ink2:#444444;
    --ink3:#888888; --accent:#2563EB; --accent2:#DBEAFE; --green:#16A34A;
    --green2:#DCFCE7; --red:#DC2626; --red2:#FEE2E2; --border:#E5E5E5;
    --shadow:0 1px 3px rgba(0,0,0,0.06),0 4px 16px rgba(0,0,0,0.04);
  }
  html { scroll-behavior:smooth; }
  body { font-family:'DM Sans',sans-serif; background:var(--cream); color:var(--ink); min-height:100vh; line-height:1.6; }

  header { background:var(--white); border-bottom:1px solid var(--border); padding:0 48px; height:64px; display:flex; align-items:center; justify-content:space-between; position:sticky; top:0; z-index:100; }
  .logo { display:flex; align-items:center; gap:10px; }
  .logo-icon { width:32px; height:32px; background:var(--accent); border-radius:8px; display:flex; align-items:center; justify-content:center; font-size:16px; }
  .logo-text { font-family:'DM Serif Display',serif; font-size:20px; color:var(--ink); }
  .header-badge { font-size:12px; color:var(--ink3); }

  .hero { padding:64px 48px 48px; max-width:1100px; margin:0 auto; }
  .hero-tag { display:inline-flex; align-items:center; gap:6px; background:var(--accent2); color:var(--accent); font-size:12px; font-weight:600; padding:4px 12px; border-radius:100px; letter-spacing:0.5px; text-transform:uppercase; margin-bottom:20px; }
  .hero h1 { font-family:'DM Serif Display',serif; font-size:clamp(36px,5vw,56px); line-height:1.1; letter-spacing:-1px; margin-bottom:16px; max-width:700px; }
  .hero h1 em { font-style:italic; color:var(--accent); }
  .hero p { font-size:16px; color:var(--ink2); max-width:520px; font-weight:300; margin-bottom:40px; }

  .pipeline { display:flex; align-items:center; flex-wrap:wrap; gap:8px; margin-bottom:48px; }
  .pipe-step { display:flex; align-items:center; gap:8px; background:var(--white); border:1px solid var(--border); border-radius:8px; padding:8px 14px; font-size:13px; font-weight:500; color:var(--ink2); }
  .pipe-dot { width:8px; height:8px; border-radius:50%; flex-shrink:0; }
  .pipe-arrow { color:var(--ink3); font-size:18px; }

  .main { max-width:1100px; margin:0 auto; padding:0 48px 80px; display:grid; grid-template-columns:360px 1fr; gap:24px; align-items:start; }

  .card { background:var(--white); border:1px solid var(--border); border-radius:16px; box-shadow:var(--shadow); overflow:hidden; }
  .card-header { padding:20px 24px 16px; border-bottom:1px solid var(--border); display:flex; align-items:center; gap:10px; }
  .card-icon { width:28px; height:28px; border-radius:6px; display:flex; align-items:center; justify-content:center; font-size:14px; flex-shrink:0; }
  .card-title { font-size:14px; font-weight:600; color:var(--ink); }
  .card-subtitle { font-size:12px; color:var(--ink3); margin-top:1px; }
  .card-body { padding:20px 24px; }

  .upload-zone { border:2px dashed var(--border); border-radius:12px; padding:32px 20px; text-align:center; cursor:pointer; transition:all 0.2s; position:relative; background:var(--cream); }
  .upload-zone:hover,.upload-zone.drag { border-color:var(--accent); background:var(--accent2); }
  .upload-zone input[type="file"] { position:absolute; inset:0; opacity:0; cursor:pointer; width:100%; height:100%; }
  .upload-icon { font-size:32px; margin-bottom:10px; }
  .upload-label { font-size:14px; font-weight:500; color:var(--ink); margin-bottom:4px; }
  .upload-hint { font-size:12px; color:var(--ink3); }
  .upload-filename { margin-top:12px; font-size:13px; color:var(--accent); font-weight:500; display:none; }

  .btn { width:100%; padding:12px; border-radius:10px; border:none; font-family:'DM Sans',sans-serif; font-size:14px; font-weight:600; cursor:pointer; transition:all 0.15s; display:flex; align-items:center; justify-content:center; gap:8px; margin-top:16px; }
  .btn-primary { background:var(--ink); color:white; }
  .btn-primary:hover { background:#333; transform:translateY(-1px); }
  .btn-primary:disabled { background:var(--border); color:var(--ink3); cursor:not-allowed; transform:none; }
  .btn-accent { background:var(--accent); color:white; }
  .btn-accent:hover { background:#1d4ed8; transform:translateY(-1px); }
  .btn-accent:disabled { background:var(--border); color:var(--ink3); cursor:not-allowed; transform:none; }

  .status-box { border-radius:10px; padding:14px 16px; font-size:13px; margin-top:16px; display:none; line-height:1.6; }
  .status-box.success { background:var(--green2); color:var(--green); border:1px solid #bbf7d0; }
  .status-box.error   { background:var(--red2);   color:var(--red);   border:1px solid #fecaca; }
  .status-box.loading { background:var(--accent2); color:var(--accent); border:1px solid #bfdbfe; }
  .status-box .stat-row { display:flex; justify-content:space-between; margin-top:8px; padding-top:8px; border-top:1px solid rgba(0,0,0,0.06); }
  .stat-item { text-align:center; }
  .stat-num { font-size:20px; font-weight:700; font-family:'DM Serif Display',serif; }
  .stat-lbl { font-size:11px; opacity:0.7; }

  .search-wrap { position:relative; margin-bottom:14px; }
  .search-icon { position:absolute; left:14px; top:50%; transform:translateY(-50%); color:var(--ink3); font-size:16px; pointer-events:none; }
  input[type="text"] { width:100%; padding:13px 14px 13px 40px; border:1.5px solid var(--border); border-radius:10px; font-family:'DM Sans',sans-serif; font-size:14px; color:var(--ink); background:var(--cream); outline:none; transition:border-color 0.15s; }
  input[type="text"]:focus { border-color:var(--accent); background:var(--white); }

  .toggle-wrap { display:flex; background:var(--cream); border:1px solid var(--border); border-radius:8px; padding:3px; margin-bottom:14px; }
  .toggle-btn { flex:1; padding:7px 4px; border:none; border-radius:6px; font-family:'DM Sans',sans-serif; font-size:12px; font-weight:500; cursor:pointer; background:transparent; color:var(--ink3); transition:all 0.15s; }
  .toggle-btn.active { background:var(--white); color:var(--ink); box-shadow:0 1px 4px rgba(0,0,0,0.08); }

  .img-query-zone { border:2px dashed var(--border); border-radius:12px; padding:20px; text-align:center; cursor:pointer; transition:all 0.2s; position:relative; background:var(--cream); margin-bottom:14px; display:none; }
  .img-query-zone:hover { border-color:var(--accent); background:var(--accent2); }
  .img-query-zone input[type="file"] { position:absolute; inset:0; opacity:0; cursor:pointer; width:100%; height:100%; }

  .results-header { display:flex; align-items:center; justify-content:space-between; padding:16px 24px; border-bottom:1px solid var(--border); }
  .results-count { font-size:13px; color:var(--ink3); }
  .results-count strong { font-size:22px; font-family:'DM Serif Display',serif; color:var(--ink); margin-right:4px; }
  .results-query { font-size:12px; background:var(--accent2); color:var(--accent); padding:3px 10px; border-radius:100px; font-weight:500; }

  .table-wrap { overflow-x:auto; padding:0 24px 24px; }
  table { width:100%; border-collapse:collapse; font-size:13px; }
  thead tr { border-bottom:1.5px solid var(--border); }
  th { padding:10px 12px; text-align:left; font-size:11px; font-weight:600; color:var(--ink3); text-transform:uppercase; letter-spacing:0.5px; white-space:nowrap; }
  td { padding:11px 12px; color:var(--ink2); border-bottom:1px solid var(--border); vertical-align:middle; }
  tr:last-child td { border-bottom:none; }
  tr:hover td { background:var(--cream); }
  .td-num { font-family:'DM Serif Display',serif; color:var(--ink3); font-size:12px; width:32px; }
  .badge { display:inline-block; padding:2px 8px; border-radius:100px; font-size:11px; font-weight:600; }
  .badge-page { background:#FEF3C7; color:#92400E; }
  .badge-line { background:#EDE9FE; color:#5B21B6; }
  .badge-word { background:#DCFCE7; color:#15803D; }
  .badge-score { background:#FEE2E2; color:#DC2626; }
  .word-highlight { font-weight:600; color:var(--ink); background:#FEF08A; padding:1px 6px; border-radius:4px; }
  .conf-bar { display:flex; align-items:center; gap:8px; }
  .conf-track { height:4px; width:48px; background:var(--border); border-radius:2px; overflow:hidden; }
  .conf-fill { height:100%; border-radius:2px; background:var(--green); }

  .empty-state { text-align:center; padding:60px 24px; color:var(--ink3); }
  .empty-icon { font-size:40px; margin-bottom:12px; }
  .empty-title { font-size:16px; font-weight:500; color:var(--ink2); margin-bottom:6px; }
  .empty-hint { font-size:13px; }

  @keyframes spin { to { transform:rotate(360deg); } }
  .spinner { width:16px; height:16px; border:2px solid currentColor; border-top-color:transparent; border-radius:50%; animation:spin 0.7s linear infinite; display:inline-block; }

  .agent-trace { background:var(--cream); border:1px solid var(--border); border-radius:10px; padding:14px 16px; font-size:12px; color:var(--ink2); margin-top:16px; display:none; line-height:1.8; font-family:monospace; }
  .trace-thought { color:#7C3AED; }
  .trace-action  { color:#2563EB; }
  .trace-observe { color:#16A34A; }

  @media (max-width:768px) {
    .main { grid-template-columns:1fr; padding:0 20px 60px; }
    .hero { padding:40px 20px 32px; }
    header { padding:0 20px; }
  }
</style>
</head>
<body>

<header>
  <div class="logo">
    <div class="logo-icon">🤖</div>
    <span class="logo-text">AgenticDocVision</span>
  </div>
  <span class="header-badge">University of Engineering &amp; Technology, Lahore</span>
</header>

<div class="hero">
  <div class="hero-tag">⚡ 5-Agent AI Pipeline</div>
  <h1>Document <em>Intelligence</em><br>at your fingertips.</h1>
  <p>Upload any research document. Our agentic AI pipeline extracts, indexes, and retrieves every word — with exact page, line, and position data.</p>
  <div class="pipeline">
    <div class="pipe-step"><span class="pipe-dot" style="background:#10B981"></span>Preprocessing</div>
    <span class="pipe-arrow">→</span>
    <div class="pipe-step"><span class="pipe-dot" style="background:#3B82F6"></span>Layout Analysis</div>
    <span class="pipe-arrow">→</span>
    <div class="pipe-step"><span class="pipe-dot" style="background:#8B5CF6"></span>OCR + Index</div>
    <span class="pipe-arrow">→</span>
    <div class="pipe-step"><span class="pipe-dot" style="background:#F59E0B"></span>LLM Correction</div>
    <span class="pipe-arrow">→</span>
    <div class="pipe-step"><span class="pipe-dot" style="background:#EF4444"></span>RAG Retrieval</div>
  </div>
</div>

<div class="main">

  <!-- LEFT PANEL -->
  <div style="display:flex;flex-direction:column;gap:20px;">

    <!-- Upload Card -->
    <div class="card">
      <div class="card-header">
        <div class="card-icon" style="background:#DCFCE7">📄</div>
        <div>
          <div class="card-title">Upload Document</div>
          <div class="card-subtitle">PDF files supported</div>
        </div>
      </div>
      <div class="card-body">
        <div class="upload-zone" id="uploadZone">
          <input type="file" accept=".pdf" id="fileInput" onchange="handleFileSelect(this)">
          <div class="upload-icon">☁️</div>
          <div class="upload-label">Drop your PDF here</div>
          <div class="upload-hint">or click to browse files</div>
          <div class="upload-filename" id="uploadFilename"></div>
        </div>
        <button class="btn btn-primary" id="processBtn" onclick="processDocument()" disabled>
          Process Document
        </button>
        <div class="status-box" id="processStatus"></div>
      </div>
    </div>

    <!-- Search Card -->
    <div class="card">
      <div class="card-header">
        <div class="card-icon" style="background:#DBEAFE">🔍</div>
        <div>
          <div class="card-title">Search</div>
          <div class="card-subtitle">Word · Line · Semantic · Image</div>
        </div>
      </div>
      <div class="card-body">

        <!-- 4 toggle buttons -->
        <div class="toggle-wrap">
          <button class="toggle-btn active" id="toggleWord"     onclick="setMode('word')">📝 Word</button>
          <button class="toggle-btn"        id="toggleLine"     onclick="setMode('line')">📄 Line</button>
          <button class="toggle-btn"        id="toggleSemantic" onclick="setMode('semantic')">🧠 Semantic</button>
          <button class="toggle-btn"        id="toggleImage"    onclick="setMode('image')">🖼️ Image</button>
        </div>

        <!-- Text input (word / line / semantic) -->
        <div class="search-wrap" id="searchWrap">
          <span class="search-icon">🔎</span>
          <input type="text" id="searchInput" placeholder="e.g. neural network, OCR..."
            onkeydown="if(event.key==='Enter') doSearch()">
        </div>

        <!-- Image upload zone (image mode only) -->
        <div class="img-query-zone" id="imageUploadZone">
          <input type="file" accept="image/*" id="queryImageInput"
                 onchange="handleQueryImage(this)">
          <div style="font-size:24px;margin-bottom:6px;">🖼️</div>
          <div class="upload-label" style="font-size:13px;">Upload word screenshot or crop</div>
          <div class="upload-hint">PNG, JPG accepted</div>
          <div class="upload-filename" id="queryImageName"></div>
        </div>

        <button class="btn btn-accent" id="searchBtn" onclick="doSearch()" disabled>
          Search
        </button>
        <div class="agent-trace" id="agentTrace"></div>
      </div>
    </div>

  </div>

  <!-- RIGHT PANEL -->
  <div class="card" id="resultsCard">
    <div class="empty-state" id="emptyState">
      <div class="empty-icon">🗂️</div>
      <div class="empty-title">No search yet</div>
      <div class="empty-hint">Upload a document and search for any word</div>
    </div>
    <div id="resultsContent" style="display:none;">
      <div class="results-header">
        <div class="results-count"><strong id="resultCount">0</strong> results found</div>
        <div class="results-query" id="resultQuery"></div>
      </div>
      <div class="table-wrap">
        <table>
          <thead>
            <tr>
              <th>#</th>
              <th>Match</th>
              <th>Document</th>
              <th>Page</th>
              <th>Line</th>
              <th>Word #</th>
              <th>Score</th>
            </tr>
          </thead>
          <tbody id="resultsBody"></tbody>
        </table>
      </div>
    </div>
  </div>

</div>

<script>
  let searchMode     = 'word';
  let selectedFile   = null;
  let queryImageFile = null;

  function setMode(mode) {
    searchMode = mode;
    ['Word','Line','Semantic','Image'].forEach(m =>
      document.getElementById('toggle'+m).classList.toggle('active', mode === m.toLowerCase())
    );
    document.getElementById('searchWrap').style.display      = mode === 'image' ? 'none'  : 'block';
    document.getElementById('imageUploadZone').style.display = mode === 'image' ? 'block' : 'none';
  }

  function handleFileSelect(input) {
    selectedFile = input.files[0];
    if (!selectedFile) return;
    document.getElementById('uploadFilename').textContent = '📎 ' + selectedFile.name;
    document.getElementById('uploadFilename').style.display = 'block';
    document.getElementById('processBtn').disabled = false;
  }

  function handleQueryImage(input) {
    queryImageFile = input.files[0];
    if (!queryImageFile) return;
    document.getElementById('queryImageName').textContent = '📎 ' + queryImageFile.name;
    document.getElementById('queryImageName').style.display = 'block';
  }

  // Drag & drop for document
  const zone = document.getElementById('uploadZone');
  zone.addEventListener('dragover', e => { e.preventDefault(); zone.classList.add('drag'); });
  zone.addEventListener('dragleave', () => zone.classList.remove('drag'));
  zone.addEventListener('drop', e => {
    e.preventDefault(); zone.classList.remove('drag');
    const f = e.dataTransfer.files[0];
    if (f && f.name.endsWith('.pdf')) {
      selectedFile = f;
      document.getElementById('uploadFilename').textContent = '📎 ' + f.name;
      document.getElementById('uploadFilename').style.display = 'block';
      document.getElementById('processBtn').disabled = false;
    }
  });

  function showStatus(id, type, html) {
    const el = document.getElementById(id);
    el.className = 'status-box ' + type;
    el.innerHTML = html;
    el.style.display = 'block';
  }

  async function processDocument() {
    if (!selectedFile) return;
    const btn = document.getElementById('processBtn');
    btn.disabled = true;
    btn.innerHTML = '<span class="spinner"></span> Processing...';
    showStatus('processStatus', 'loading', '⏳ Running 4 agents (preprocess → layout → OCR → LLM correction) — this may take 1-2 minutes...');
    const form = new FormData();
    form.append('file', selectedFile);
    try {
      const res  = await fetch('/upload', { method: 'POST', body: form });
      const data = await res.json();
      if (data.success) {
        showStatus('processStatus', 'success',
          `✅ <strong>Document processed!</strong>
          <div class="stat-row">
            <div class="stat-item"><div class="stat-num">${data.pages}</div><div class="stat-lbl">Pages</div></div>
            <div class="stat-item"><div class="stat-num">${data.word_count}</div><div class="stat-lbl">Words Indexed</div></div>
          </div>`);
        document.getElementById('searchBtn').disabled = false;
        btn.innerHTML = '✅ Processed';
      } else {
        showStatus('processStatus', 'error', '❌ ' + data.error);
        btn.disabled = false; btn.innerHTML = 'Process Document';
      }
    } catch(e) {
      showStatus('processStatus', 'error', '❌ ' + e.message);
      btn.disabled = false; btn.innerHTML = 'Process Document';
    }
  }

  async function doSearch() {
    const btn   = document.getElementById('searchBtn');
    const trace = document.getElementById('agentTrace');
    btn.disabled = true;
    btn.innerHTML = '<span class="spinner"></span> Searching...';
    trace.style.display = 'block';

    try {
      // ── IMAGE MODE ────────────────────────────────────
      if (searchMode === 'image') {
        if (!queryImageFile) {
          trace.innerHTML = '<div class="trace-thought">❌ Please upload a query image first.</div>';
          btn.disabled = false; btn.innerHTML = 'Search'; return;
        }
        trace.innerHTML = `
          <div class="trace-line trace-thought">💭 Thought: Image query — reading text from the image (OCR)</div>
          <div class="trace-line trace-action">⚡ Action: OCR the crop, then search the document...</div>`;
        const form = new FormData();
        form.append('file', queryImageFile);
        const res  = await fetch('/image-search', { method: 'POST', body: form });
        const data = await res.json();
        if (data.success) {
          const stype = data.search_type || 'image';
          if (data.extracted_text) {
            trace.innerHTML += `
              <div class="trace-line trace-observe">👁 Observation: Recognized text → "${data.extracted_text}"</div>
              <div class="trace-line trace-observe">✅ Done — ${data.total_found} match(es) found in document</div>`;
          } else {
            trace.innerHTML += `
              <div class="trace-line trace-observe">👁 Observation: No readable text — fell back to ORB visual matching</div>
              <div class="trace-line trace-observe">✅ Done — ${data.total_found} matching region(s) found</div>`;
          }
          const q = data.extracted_text
                    ? `${data.extracted_text} (from image)`
                    : queryImageFile.name;
          renderResults({ query: q, search_type: stype,
                          total_found: data.total_found, results: data.results });
        } else {
          trace.innerHTML += `<div style="color:red">❌ ${data.error}</div>`;
        }

      // ── SEMANTIC MODE ─────────────────────────────────
      } else if (searchMode === 'semantic') {
        const query = document.getElementById('searchInput').value.trim();
        if (!query) { btn.disabled = false; btn.innerHTML = 'Search'; return; }
        trace.innerHTML = `
          <div class="trace-line trace-thought">💭 Thought: Semantic query — computing embedding for "${query}"</div>
          <div class="trace-line trace-action">⚡ Action: Comparing against ${' '}stored embeddings in ChromaDB...</div>`;
        const form = new FormData();
        form.append('query', query);
        form.append('search_type', 'semantic');
        const res  = await fetch('/search', { method: 'POST', body: form });
        const data = await res.json();
        if (data.success) {
          trace.innerHTML += `
            <div class="trace-line trace-observe">👁 Observation: Cosine similarity computed</div>
            <div class="trace-line trace-observe">✅ Done — ${data.total_found} semantically similar result(s)</div>`;
          renderResults(data);
        } else {
          trace.innerHTML += `<div style="color:red">❌ ${data.error}</div>`;
        }

      // ── WORD / LINE MODE ──────────────────────────────
      } else {
        const query = document.getElementById('searchInput').value.trim();
        if (!query) { btn.disabled = false; btn.innerHTML = 'Search'; return; }
        const modeVerb = searchMode === 'line'
          ? 'grouping indexed words into lines'
          : 'exact + fuzzy matching against indexed words';
        trace.innerHTML = `
          <div class="trace-line trace-thought">💭 Thought: ${searchMode === 'line' ? 'Line' : 'Word'} search for "${query}"</div>
          <div class="trace-line trace-action">⚡ Action: ${modeVerb} in ChromaDB...</div>`;
        const form = new FormData();
        form.append('query', query);
        form.append('search_type', searchMode);
        const res  = await fetch('/search', { method: 'POST', body: form });
        const data = await res.json();
        if (data.success) {
          trace.innerHTML += `
            <div class="trace-line trace-observe">👁 Observation: ChromaDB word index searched</div>
            <div class="trace-line trace-observe">✅ Done — ${data.total_found} ${searchMode === 'line' ? 'line(s)' : 'occurrence(s)'} found</div>`;
          renderResults(data);
        } else {
          trace.innerHTML += `<div style="color:red">❌ ${data.error}</div>`;
        }
      }
    } catch(e) {
      trace.innerHTML += `<div style="color:red">❌ ${e.message}</div>`;
    }

    btn.disabled = false; btn.innerHTML = 'Search';
  }

  function renderResults(data) {
    document.getElementById('emptyState').style.display     = 'none';
    document.getElementById('resultsContent').style.display = 'block';
    document.getElementById('resultCount').textContent      = data.total_found;
    document.getElementById('resultQuery').textContent      = '"' + data.query + '"';

    const tbody = document.getElementById('resultsBody');
    if (data.total_found === 0) {
      tbody.innerHTML = `<tr><td colspan="7" style="text-align:center;padding:40px;color:#888">
        No results found.</td></tr>`;
      return;
    }

    const dash = '<span style="color:#bbb">—</span>';

    tbody.innerHTML = data.results.map((r, i) => {
      const isImage    = data.search_type === 'image';
      const isLine     = data.search_type === 'line';
      const isSemantic = data.search_type === 'semantic';

      // ── Match column ──────────────────────────────────────────────
      const wordCell   = isImage    ? '🖼️ visual match'
                       : isLine     ? `<span style="color:#444">${r.word}</span>`
                       : isSemantic ? `<span class="word-highlight">${r.word}</span> <span style="font-size:10px;color:#888">(semantic)</span>`
                       :              `<span class="word-highlight">${r.word}</span>`;

      // ── Line / Word# columns (hide values that don't apply) ───────
      const lineCell = (r.line_no && r.line_no > 0)
                       ? `<span class="badge badge-line">l.${r.line_no}</span>` : dash;
      const wordNo   = isImage ? `<span class="badge badge-score">s.${r.score}</span>`
                       : (r.word_no && r.word_no > 0)
                         ? `<span class="badge badge-word">w.${r.word_no}</span>` : dash;

      // ── Score column ──────────────────────────────────────────────
      // Relevance (similarity / visual match) for semantic & image;
      // OCR confidence for exact word / line matches.
      const useScore  = isImage || isSemantic;
      const rawVal    = useScore ? (r.score * 100) : (r.confidence || 0);
      const confVal   = Math.max(0, Math.min(100, rawVal));
      const confLabel = confVal.toFixed(0) + '%';
      const scoreClr  = useScore ? 'var(--accent)' : 'var(--green)';

      return `<tr>
        <td class="td-num">${i+1}</td>
        <td>${wordCell}</td>
        <td style="font-size:12px;color:#666;max-width:120px;overflow:hidden;
            text-overflow:ellipsis;white-space:nowrap" title="${r.book}">${r.book}</td>
        <td><span class="badge badge-page">p.${r.page_no}</span></td>
        <td>${lineCell}</td>
        <td>${wordNo}</td>
        <td>
          <div class="conf-bar">
            <div class="conf-track">
              <div class="conf-fill" style="width:${confVal}%;background:${scoreClr}"></div>
            </div>
            <span style="font-size:11px;color:#666">${confLabel}</span>
          </div>
        </td>
      </tr>`;
    }).join('');
  }
</script>
</body>
</html>"""


@app.get("/", response_class=HTMLResponse)
async def root():
    return HTML


# ══════════════════════════════════════════════════════════════════════
# LAUNCH
# ══════════════════════════════════════════════════════════════════════
def start_server():
    # Direct uvicorn.run ki bajaye config object banayein
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="warning")
    server = uvicorn.Server(config)

    # Isko loop_factory ke bagair run karne ke liye async method call karein
    import asyncio
    asyncio.run(server.serve())

thread = threading.Thread(target=start_server, daemon=True)
thread.start()

import time; time.sleep(2)

ngrok.set_auth_token(os.getenv("NGROK_AUTH_TOKEN"))
public_url = ngrok.connect(8000)
print()
print("=" * 55)
print("🚀 AgenticDocVision is LIVE!")
print(f"🌐 Open this URL: {public_url}")
print("=" * 55)

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use



🚀 AgenticDocVision is LIVE!
🌐 Open this URL: NgrokTunnel: "https://uncanny-graceless-hut.ngrok-free.dev" -> "http://localhost:8000"
